# FinTAG-RAG — Financial Question Answering, Colab Demo

**Temporal-Aware Hybrid Retrieval and Symbolic Reasoning Framework**, implementing
the six-module architecture from the thesis / paper *FinTAG-RAG: A Temporal-Aware
Hybrid Retrieval and Reasoning Framework for Robust Financial Question Answering*:

1. **Query Processing** — financial ontology expansion, entity/temporal normalization
2. **Hybrid Retrieval** — dense (FAISS) + sparse (BM25) + Reciprocal Rank Fusion + Cross-Encoder rerank
3. **Context Filtering** — relevance threshold + near-duplicate removal
4. **Temporal Compatibility** — discards passages from the wrong fiscal year
5. **Symbolic Reasoning** — deterministic arithmetic over verified operands (no LLM hallucinated math)
6. **Answer Generation** — LLM synthesizes the final answer from verified evidence + computed result

Plus a **causal reasoning** extension (cue-phrase cause/effect extraction), giving
four reported accuracy dimensions: **context, temporal, numerical, causal**.

Dataset: [FinQA](https://github.com/czyssrs/FinQA) (Chen et al., 2021).

**Runtime:** Google Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).
Everything here is self-contained in the `fintag_rag/` folder of the `Finrag` repo
and does **not** depend on the rest of that repo.

| Cell block | What it does | Time |
|---|---|---|
| Setup (1-4) | Install deps, clone repo, load FinQA, verify GPU | ~4 min |
| Corpus + retriever (5) | Build the shared multi-document retrieval index | ~3-6 min |
| Module walkthrough (6-10) | Inspect each of the 6 modules on one example | ~1 min |
| Full pipeline (11-13) | Load the LLM, run one question end-to-end | ~2 min |
| Evaluation (14-18) | Live baseline comparison + 4-dim accuracy report | ~15-30 min |
| Save/reload (19-21) | Persist to Google Drive; reload without rebuilding | ~1 min |
| Committee demo (22-25) | Scorecard, scripted walkthrough, live Q&A, Gradio app | ~2 min |


In [ ]:
# ── Cell 1: Install dependencies ───────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

print('PyTorch...')
pip('torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121')

print('Retrieval + LLM packages...')
pip(
    'sentence-transformers>=2.2.0', 'faiss-cpu>=1.7.4', 'rank-bm25>=0.2.2',
    'transformers>=4.40.0', 'accelerate>=0.28.0',
    'numpy', 'scikit-learn', 'tqdm', 'requests', 'matplotlib',
)
print('Done.')


In [ ]:
# ── Cell 2: Clone the repo and jump into the self-contained fintag_rag/ project ─
import os, subprocess, sys

REPO   = 'https://github.com/Stuti012/Finrag.git'
BRANCH = 'claude/thesis-qa-system-4tg08s'   # ← update if this lands on a different branch
DEST   = 'Finrag'

if not os.path.isdir(DEST):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, DEST], check=True)
else:
    subprocess.run(['git', '-C', DEST, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', DEST, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', DEST, 'pull', 'origin', BRANCH], check=True)

PROJECT_DIR = os.path.join(DEST, 'fintag_rag')
os.chdir(PROJECT_DIR)
if '.' not in sys.path:
    sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')

# Optional: OpenAI API key, to use GPT-3.5-Turbo for answer generation exactly as
# in the thesis (Table 4.1). Leave blank to use the free local model instead.
OPENAI_API_KEY = ''   # ← paste your key here, or leave blank
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], check=True)
    print('OpenAI API key set -- answer generation will use GPT-3.5-Turbo.')
else:
    print('No OpenAI key -- answer generation will use a local open model (Qwen2.5-1.5B-Instruct).')


In [ ]:
# ── Cell 3: Download and load all three FinQA splits ───────────────────────────
from src.data import load_finqa_dataset

dataset = load_finqa_dataset('./finqa_data', download=True)
train_examples = dataset.get('train', [])
dev_examples   = dataset.get('dev', [])
test_examples  = dataset.get('test', [])

print(f'train : {len(train_examples):>5}')
print(f'dev   : {len(dev_examples):>5}')
print(f'test  : {len(test_examples):>5}')

ex0 = test_examples[0]
print(f'\nSample question: {ex0.question}')
print(f'Gold answer    : {ex0.answer}')


In [ ]:
# ── Cell 4: Verify GPU ──────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')
else:
    print('No GPU detected -- Runtime -> Change runtime type -> T4 GPU. '
          'The pipeline still runs on CPU, just slower.')


## Build the Shared Retrieval Corpus

Following Section 3.4 of the thesis, we pool chunks from **many documents**
into one shared index -- not just the document a question was originally
paired with -- so retrieval has to contend with the same cross-year,
cross-company ambiguity that motivates the whole architecture. The test
questions' own source documents are always included; a configurable number
of additional "distractor" documents from the train/dev splits are added on
top of that.


In [ ]:
# ── Cell 5: Build the corpus + hybrid retriever ─────────────────────────────────
from src.config import FinTAGRAGConfig
from src.data import build_corpus
from src.retrieval import HybridRetriever

N_DISTRACTOR_TRAIN = 700   # ← increase for a harder/more realistic corpus (slower to build)
N_DISTRACTOR_DEV    = 150

corpus_examples = test_examples + train_examples[:N_DISTRACTOR_TRAIN] + dev_examples[:N_DISTRACTOR_DEV]
config = FinTAGRAGConfig()
chunks = build_corpus(corpus_examples, config.chunk_size_tokens, config.chunk_overlap_tokens)
print(f'Corpus: {len(corpus_examples)} documents -> {len(chunks)} chunks '
      f'({sum(1 for c in chunks if c.kind == "table")} table / '
      f'{sum(1 for c in chunks if c.kind == "text")} text)')

print('\nBuilding FAISS dense index + BM25 sparse index (downloads the embedding '
      'and cross-encoder models on first run)...')
retriever = HybridRetriever(chunks, config)
print('✓ Retriever ready.')


## Module-by-module Walkthrough

Inspect each of the six stages on a single example before running the full pipeline.

In [ ]:
# ── Cell 6: Query Processing Module (Section 3.5.1) ────────────────────────────
from src.ontology import QueryProcessor

qp = QueryProcessor()
sample_question = test_examples[0].question
enriched = qp.process(sample_question)

print(f'Original         : {enriched.original}')
print(f'Normalized       : {enriched.normalized}')
print(f'Expanded terms   : {enriched.expanded_terms}')
print(f'Explicit years   : {enriched.explicit_years}')
print(f'Implicit refs    : {enriched.implicit_temporal_refs}')
print(f'Operation hint   : {enriched.operation_hint}')


In [ ]:
# ── Cell 7: Hybrid Retrieval + Context Filtering + Temporal Compatibility ──────
from src.retrieval import filter_context
from src.temporal import TemporalCompatibilityModule

retrieved = retriever.retrieve(enriched)
print(f'Retrieved (post rerank) : {len(retrieved)} chunks')
for r in retrieved[:5]:
    print(f'  score={r.score:.3f}  years={r.chunk.fiscal_years}  {r.chunk.text[:90]}...')

context_filtered = filter_context(retrieved, retriever, config)
print(f'\nAfter context filtering (tau={config.context_relevance_threshold}): {len(context_filtered)} chunks')

temporal_module = TemporalCompatibilityModule(config)
temporally_filtered = temporal_module.filter(enriched, context_filtered)
print(f'After temporal filtering (delta={config.temporal_compatibility_threshold}): {len(temporally_filtered)} chunks')
for r in temporally_filtered:
    print(f'  years={r.chunk.fiscal_years}  {r.chunk.text[:90]}...')


In [ ]:
# ── Cell 8: Symbolic Reasoning Module (Section 3.5.5) ──────────────────────────
from src.symbolic import SymbolicReasoner

symbolic_result = SymbolicReasoner(config).reason(enriched, temporally_filtered)
print(f'Success   : {symbolic_result.success}')
print(f'Operation : {symbolic_result.operation}')
print(f'Value     : {symbolic_result.value}')
print('Trace:')
for step in symbolic_result.trace:
    print(f'  {step}')
print(f'\nGold answer: {test_examples[0].answer}')


In [ ]:
# ── Cell 9: Causal Reasoning extension ──────────────────────────────────────────
from src.causal import CausalExtractor, evaluate_causal_accuracy, is_causal_question

causal_demo_sentence = 'Operating margin declined due to rising raw material costs and higher freight expenses.'
rel = CausalExtractor().extract_from_sentence(causal_demo_sentence)
print(f'Sentence : {causal_demo_sentence}')
if rel:
    print(f'Cause    : {rel.cause}')
    print(f'Effect   : {rel.effect}')
    print(f'Cue      : {rel.cue_phrase}  (confidence={rel.confidence:.2f})')

print(f'\nIs the sample question causal? {is_causal_question(sample_question)}')

causal_report = evaluate_causal_accuracy()
print(f"\nCausal accuracy (hand-labeled set, n={causal_report['n_examples']}):")
print(f"  Detection rate  : {causal_report['detection_rate']:.1%}")
print(f"  Mean span F1    : {causal_report['mean_span_f1']:.1%}")


## Full Pipeline + Answer Generation

In [ ]:
# ── Cell 10: Build the full pipeline (loads the LLM on first use) ──────────────
from src.pipeline import FinTAGRAGPipeline

config.use_openai = bool(os.environ.get('OPENAI_API_KEY'))
pipeline = FinTAGRAGPipeline(retriever=retriever, config=config, load_generator=True)
print(f'Pipeline ready. Answer generation backend: '
      f'{"OpenAI " + config.openai_model if config.use_openai else config.llm_model}')


In [ ]:
# ── Cell 11: Run one question through the full FinTAG-RAG pipeline ─────────────
example = test_examples[0]
result = pipeline.answer(example.question, mode='full')

print('━' * 70)
print(f'Question : {example.question}')
print(f'Gold     : {example.answer}')
print('━' * 70)
print(f'Retrieved -> {len(result.retrieved)}  |  '
      f'Context-filtered -> {len(result.context_filtered)}  |  '
      f'Temporally-filtered -> {len(result.temporally_filtered)}')
if result.symbolic_result:
    print(f'\nSymbolic reasoning: {result.symbolic_result.operation} -> {result.symbolic_result.value}')
    for step in result.symbolic_result.trace:
        print(f'  {step}')
print(f'\nFinal answer: {result.answer_text}')


## Evaluation — Reproducing the Thesis's Baseline Comparison Live

The thesis (Table 5.1-5.4) compares BM25, Dense Retrieval, LLM-only, and
Hybrid RAG (FinTAG-RAG). The cells below **recompute these numbers on this
run** against a sample of the FinQA test set, rather than reproducing the
numbers printed in the paper -- so what you see is a genuine, if
smaller-sample, reproduction.


In [ ]:
# ── Cell 12: Baseline comparison (BM25 / Dense / LLM-only / FinTAG-RAG) ─────────
from src.evaluation import run_baseline_comparison

EVAL_SAMPLE_SIZE = 100   # ← increase for a more reliable estimate (slower)

baseline_results = run_baseline_comparison(
    pipeline, test_examples, max_examples=EVAL_SAMPLE_SIZE,
    modes=['bm25', 'dense', 'llm_only', 'full'],
)


In [ ]:
# ── Cell 13: Comparison plots (reproducing thesis Figs. 2-4, computed live) ─────
import matplotlib.pyplot as plt
import numpy as np

mode_labels = {'bm25': 'BM25', 'dense': 'Dense', 'llm_only': 'LLM-only', 'full': 'FinTAG-RAG'}
modes_order = ['bm25', 'dense', 'llm_only', 'full']
colors = {'bm25': '#F4A261', 'dense': '#2A9D8F', 'llm_only': '#457B9D', 'full': '#2E7D32'}

metrics_to_plot = [
    ('context_precision', 'Context Precision'),
    ('temporal_alignment', 'Temporal Alignment'),
    ('numerical_accuracy', 'Numerical Accuracy'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (key, title) in zip(axes, metrics_to_plot):
    vals = [baseline_results[m]['metrics'][key] for m in modes_order]
    bars = ax.bar([mode_labels[m] for m in modes_order], vals, color=[colors[m] for m in modes_order])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f'{v:.0%}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 1.05)
plt.suptitle('FinTAG-RAG vs. Baselines — Live Reproduction on FinQA Test Sample', fontweight='bold')
plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Numeric accuracy + MAE (thesis Fig. 3 style)
fig, ax1 = plt.subplots(figsize=(7, 4.5))
acc_vals = [baseline_results[m]['metrics']['numerical_accuracy'] for m in modes_order]
mae_vals = [baseline_results[m]['metrics']['mae'] for m in modes_order]
x = np.arange(len(modes_order))
ax1.bar(x - 0.2, acc_vals, width=0.4, color='#1565C0', label='Numerical accuracy')
ax1.set_ylabel('Numerical accuracy')
ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.bar(x + 0.2, mae_vals, width=0.4, color='#E76F51', label='MAE (lower is better)')
ax2.set_ylabel('Mean absolute error')
ax1.set_xticks(x)
ax1.set_xticklabels([mode_labels[m] for m in modes_order])
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2)
plt.title('Numeric Reasoning: Accuracy vs. MAE', fontweight='bold', pad=30)
plt.tight_layout()
plt.savefig('outputs/figures/numeric_accuracy_mae.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Cell 14: Full 4-dimension summary report ────────────────────────────────────
import json, os
from src.evaluation import full_four_dimension_report

os.makedirs('outputs', exist_ok=True)
report = full_four_dimension_report(pipeline, test_examples, max_examples=EVAL_SAMPLE_SIZE)

print('=' * 62)
print('FinTAG-RAG — FOUR-DIMENSION ACCURACY REPORT (live, this run)')
print('=' * 62)
print(f"Context precision   : {report['context_precision']:.1%}")
print(f"Context recall      : {report['context_recall']:.1%}")
print(f"Temporal alignment  : {report['temporal_alignment']:.1%}")
print(f"Numerical accuracy  : {report['numerical_accuracy']:.1%}")
print(f"Numerical MAE       : {report['mae']:.2f}")
print(f"Causal detection    : {report['causal_detection_rate']:.1%}  (n={report['causal_eval']['n_examples']})")
print(f"Causal span F1      : {report['causal_span_f1']:.1%}")
print('=' * 62)

with open('outputs/evaluation_report.json', 'w') as f:
    json.dump({k: v for k, v in report.items() if k not in ('full_eval', 'causal_eval')}, f, indent=2, default=str)
print("Saved -> outputs/evaluation_report.json")


## Save the Model for the Committee Demo

Persists the FAISS/BM25 index, the corpus, and the config to Google Drive so
a fresh Colab session can jump straight to the demo without rebuilding the
retrieval index (the LLM itself is downloaded fresh each session from
Hugging Face, which is fast; only the *index* is expensive to rebuild).

In [ ]:
# ── Cell 15: Save the retrieval index + corpus + config to Google Drive ─────────
import pickle, json, datetime

SAVE_TO_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/FinTAG_RAG_saved_model'

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(DRIVE_DIR, exist_ok=True)

    with open(os.path.join(DRIVE_DIR, 'chunks.pkl'), 'wb') as f:
        pickle.dump(retriever.chunks, f)
    np = __import__('numpy')
    np.save(os.path.join(DRIVE_DIR, 'embeddings.npy'), retriever.embeddings)
    with open(os.path.join(DRIVE_DIR, 'bm25_corpus.pkl'), 'wb') as f:
        pickle.dump(retriever._bm25_corpus, f)

    meta = {
        'config': config.__dict__,
        'n_chunks': len(retriever.chunks),
        'n_corpus_docs': len(corpus_examples),
        'saved_at': datetime.datetime.utcnow().isoformat(),
    }
    with open(os.path.join(DRIVE_DIR, 'run_metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    for item in ('outputs/evaluation_report.json', 'outputs/figures'):
        if os.path.exists(item):
            dest = os.path.join(DRIVE_DIR, os.path.basename(item))
            if os.path.isdir(item):
                shutil = __import__('shutil')
                shutil.copytree(item, dest, dirs_exist_ok=True)
            else:
                shutil = __import__('shutil')
                shutil.copy2(item, dest)

    print(f'✓ Saved retrieval index + config + reports -> {DRIVE_DIR}')
else:
    print('SAVE_TO_DRIVE=False -- nothing persisted.')


In [ ]:
# ── Cell 16: Reload the saved index in a fresh Colab session ────────────────────
# Run this cell after Cells 1-4 (installs, repo clone, dataset load, GPU check)
# in a *new* Colab session to skip rebuilding the FAISS/BM25 index from scratch.

RELOAD_FROM_DRIVE = True

if RELOAD_FROM_DRIVE:
    import pickle, json, numpy as np
    from google.colab import drive
    from src.config import FinTAGRAGConfig
    from src.retrieval import HybridRetriever

    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = '/content/drive/MyDrive/FinTAG_RAG_saved_model'
    meta_path = os.path.join(DRIVE_DIR, 'run_metadata.json')

    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        config = FinTAGRAGConfig(**{k: v for k, v in meta['config'].items() if k in FinTAGRAGConfig.__dataclass_fields__})

        with open(os.path.join(DRIVE_DIR, 'chunks.pkl'), 'rb') as f:
            saved_chunks = pickle.load(f)

        # Rebuild the retriever object without re-encoding (reuse cached embeddings).
        retriever = HybridRetriever.__new__(HybridRetriever)
        retriever.chunks = saved_chunks
        retriever.config = config
        retriever._embedder = None
        retriever._cross_encoder = None
        retriever._show_progress = False
        retriever.embeddings = np.load(os.path.join(DRIVE_DIR, 'embeddings.npy'))
        import faiss
        dim = retriever.embeddings.shape[1]
        retriever.index = faiss.IndexFlatIP(dim)
        retriever.index.add(retriever.embeddings)
        with open(os.path.join(DRIVE_DIR, 'bm25_corpus.pkl'), 'rb') as f:
            retriever._bm25_corpus = pickle.load(f)
        from rank_bm25 import BM25Okapi
        retriever.bm25 = BM25Okapi(retriever._bm25_corpus)

        from src.pipeline import FinTAGRAGPipeline
        pipeline = FinTAGRAGPipeline(retriever=retriever, config=config, load_generator=True)
        print(f'✓ Reloaded {len(retriever.chunks)} chunks and rebuilt the pipeline '
              f'without re-encoding.')
    else:
        print('No saved run found in Drive -- run Cell 5 (build) then Cell 15 (save) first.')


## Committee Demo

A per-question scorecard covering all four accuracy dimensions, a scripted
walkthrough of curated questions, a live free-form question cell, and an
optional shareable Gradio web app.

In [ ]:
# ── Cell 17: Four-dimension scorecard for a single test question ───────────────
from src.evaluation import evaluate_example_with_trace

def print_scorecard(example, mode='full'):
    record, result = evaluate_example_with_trace(pipeline, example, mode=mode)
    bar = lambda v: ('█' * int(round(v * 20)) + '░' * (20 - int(round(v * 20)))) if v is not None else '░' * 20
    fmt = lambda v: f'{v:.0%}' if v is not None else 'n/a'

    print('━' * 72)
    print(f'Q: {example.question}')
    print('━' * 72)
    print(f'Predicted : {result.numeric_answer}')
    print(f'Gold      : {example.answer}   {"✓" if record.is_correct else ("✗" if record.is_correct is False else "")}')
    print(f'Answer    : {result.answer_text}')
    print()
    print(f"  Context   [{bar(record.context_precision)}] {fmt(record.context_precision)}  "
          f"({record.n_evidence} passages, recall={fmt(record.context_recall)})")
    print(f"  Temporal  [{bar(record.temporal_alignment)}] {fmt(record.temporal_alignment)}")
    print(f"  Numerical [{bar(1.0 if record.is_correct else 0.0 if record.is_correct is not None else None)}] "
          f"{'correct' if record.is_correct else ('incorrect' if record.is_correct is False else 'n/a')}")
    if result.causal_relations:
        top = result.causal_relations[0]
        print(f"  Causal    [{bar(top.confidence)}] {fmt(top.confidence)}  ({top.cause} -> {top.effect})")
    else:
        print(f'  Causal    [{bar(None)}] n/a  (not a causal question)')
    print('━' * 72)
    return record, result

_ = print_scorecard(test_examples[0])


In [ ]:
# ── Cell 18: Scripted committee walkthrough — a handful of curated questions ────
import random
random.seed(config.seed)

demo_examples = random.sample(test_examples, k=min(5, len(test_examples)))
for ex in demo_examples:
    print_scorecard(ex)
    print()


In [ ]:
# ── Cell 19: Interactive — ask a question live in front of the committee ───────
# A) pick any test-set question by index (has a gold answer to check against)
# B) ask a fully custom question against your own table + narrative text

MODE = 'A'   # 'A' or 'B'

if MODE == 'A':
    idx = 0  # ← change this and re-run to try a different question
    print_scorecard(test_examples[idx])

elif MODE == 'B':
    result = pipeline.answer(
        'What was the percentage increase in revenue from 2022 to 2024?',
        mode='full',
    )
    print(f'Answer: {result.answer_text}')
    if result.symbolic_result:
        print(f'Computation trace:')
        for step in result.symbolic_result.trace:
            print(f'  {step}')
    # Note: mode B only works if a chunk for the years/metric you ask about
    # already exists in the corpus built in Cell 5. To ask about an entirely
    # new company/table, add it to `corpus_examples` and re-run Cell 5.


### Optional — Shareable Gradio Web App

In [ ]:
# ── Cell 20: Optional Gradio web UI for a polished live demo ────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True)
import gradio as gr

def gradio_answer(question):
    result = pipeline.answer(question, mode='full')
    lines = [f'**Answer:** {result.answer_text}', '']
    if result.symbolic_result:
        lines.append(f'**Computation:** `{result.symbolic_result.operation}` -> `{result.symbolic_result.value}`')
        lines += [f'- {s}' for s in result.symbolic_result.trace]
    lines.append('')
    lines.append(f'Retrieved {len(result.retrieved)} -> context-filtered {len(result.context_filtered)} '
                 f'-> temporally-filtered {len(result.temporally_filtered)} passages.')
    if result.causal_relations:
        lines.append('')
        lines.append('**Causal relations found:**')
        for r in result.causal_relations[:3]:
            lines.append(f'- {r.cause} → {r.effect} (confidence {r.confidence:.2f})')
    return '\n'.join(lines)

demo = gr.Interface(
    fn=gradio_answer,
    inputs=gr.Textbox(label='Question', value=test_examples[0].question, lines=2),
    outputs=gr.Markdown(label='FinTAG-RAG Answer'),
    title='FinTAG-RAG — Committee Demo',
    description='Ask a question over the FinQA corpus built in this session. '
                'Reports the retrieval/temporal/symbolic reasoning trace alongside the answer.',
    examples=[[e.question] for e in test_examples[:5]],
)
demo.launch(share=True, debug=False)
